# Spark

Hoewel het MapReduce algoritme een aantal voordelen heeft is het echter niet het meest efficiente algoritme dat mogelijk is.
Omdat alles ingelezen wordt vanaf de harde schijf, tussenresultaten op de schijf opgeslagen worden en de finale resultaten ook wordt er tot wel 90% van de rekentijd gespendeerd in lees- of schrijfopdrachten.

Spark is geintroduceerd om dit te versnellen door gebruik te maken van in-memory processing.
Hierdoor is Spark tot 3 keer sneller op grote datasets en tot 100 keer op kleinere datasets.

Het spark framework kan gebruik maken van een externe opslag-locatie voor bestanden bij te houden (zoals MinIO) en bestaat uit de volgende componenten:
* SparkCore
* Spark SQL
* Spark Streaming
* MLlib
* SparkGraph

Daarnaast zijn er ook verschillende Spark Api's voor verschillende programmeertalen zoals Python, Scala, Java, ...
Hierdoor is het framework ook flexibeler dan het standaard MapReduce algoritme.
Heel veel informatie over het spark framework vind je in de [documentatie](https://spark.apache.org/docs/latest/quick-start.html) en de programming guides (bovenaan).

In [1]:
from minio import Minio
import os

client = Minio(
    endpoint="minio1:9000",
    access_key="bigdata",
    secret_key="bigdata123",
    secure=False,
    region='eu-west-1'
)

# Check verbinding
try:
    print(client.list_buckets())
    print("Verbonden met MinIO")
except Exception as e:
    print("Fout bij verbinden:", e)

bucket_name = "02-spark"
keep_files = ["input.txt", "titanic.csv", "input.csv"]

# Stap 1: bucket aanmaken als die niet bestaat
if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' aangemaakt")

else:
    print(f"Bucket '{bucket_name}' bestaat al")

    # Stap 2: bucket leegmaken behalve keep1.txt en keep2.txt
    for obj in client.list_objects(bucket_name):
        if obj.object_name not in keep_files:
            client.remove_object(bucket_name, obj.object_name)
            print(f"Verwijderd: {obj.object_name}")


# Stap 3: check files in keep_files
for fname in keep_files:
    # Upload bestand uit lokale map als het ontbreekt
    if os.path.exists(fname):
        client.fput_object(bucket_name, fname, fname)
        print(f"Geüpload: {fname}")
    else:
        print(f"Let op: {fname} bestaat niet lokaal en kon niet geüpload worden.")

[Bucket(name='01-mapreduce', creation_date=datetime.datetime(2026, 3, 3, 13, 11, 43, 563000, tzinfo=datetime.timezone.utc)), Bucket(name='02-spark', creation_date=datetime.datetime(2026, 3, 10, 13, 11, 25, 375000, tzinfo=datetime.timezone.utc)), Bucket(name='mijnclibucket', creation_date=datetime.datetime(2026, 3, 6, 13, 6, 30, 467000, tzinfo=datetime.timezone.utc)), Bucket(name='mijnpythonbucket', creation_date=datetime.datetime(2026, 2, 24, 14, 8, 57, 985000, tzinfo=datetime.timezone.utc))]
Verbonden met MinIO
Bucket '02-spark' bestaat al
Verwijderd: iris.csv
Verwijderd: titanic.parquet/
Verwijderd: titanic_parquet/
Geüpload: input.txt
Geüpload: titanic.csv
Let op: input.csv bestaat niet lokaal en kon niet geüpload worden.


## Installatie

We gebruiken het pyspark package om distributed applicaties te schrijven.
In de cluster omgeving zijn er 4 nodes die gebruik worden als pyspark cluster.

* Een spark-master: Beheer van Spark applicaties, verdelen over de beschikbare nodes en fout-tolerantie. De webUI van de spark-master is bereikbaar via: http://localhost:8080/
* 3 spark nodes: Uitvoeren van spark-applicaties

## Connecteren met de spark-cluster

Een **SparkContext** is de oorspronkelijke ingang tot Spark en laat je werken met RDD's en clusterresources beheren. 
Vanaf Spark 2.0 wordt echter de **SparkSession** geïntroduceerd als een hogere abstractie, die zowel de 
`SparkContext` als de SQLContext, HiveContext en StreamingContext combineert in één ingangspunt. 

**Belangrijk:** SparkSession bevat altijd een interne SparkContext, zodat je nog steeds toegang kan hebben tot de lage-niveau API’s indien nodig.


In [2]:
from pyspark.sql import SparkSession

# dit moet je niet vanbuiten kennen en zal ik je altijd geven
spark = SparkSession.builder \
    .appName("SparkMinIODemo") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.301") \
    .config("spark.hadoop.fs.s3a.access.key", "bigdata") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

sc = spark.sparkContext
print(sc)
print(spark)


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-352a81da-9cc7-42f0-a180-e8e5bff5e4a8;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.301 in central
:: resolution report :: resolve 168ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.301 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 by [com.amazonaws#aws-java-sdk-bundle;1.12.301] in [default]
	---------------------------------------------------------------------
	|      

<SparkContext master=spark://spark-master:7077 appName=SparkMinIODemo>


26/03/13 15:42:19 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## RDD vs DataFrames

Een **RDD (Resilient Distributed Dataset)** is de fundamentele datastructuur in Spark en biedt lage-niveau 
transformaties en acties. Het is flexibel, maar minder geoptimaliseerd.

Een **DataFrame** is een hoger-niveau API die tabulaire data met schema’s beheert. 
DataFrames zijn sterk geoptimaliseerd via de Catalyst optimizer en Tungsten engine. 
In de praktijk gebruik je meestal DataFrames voor productietaken, en RDD’s voor speciale berekeningen.


In [6]:
# Voorbeeld RDD
rdd = sc.parallelize([('a', 1), ('b', 1), ('a', 2)])
print('rdd collect:', rdd.collect()) # collect download de data van de cluster naar lokaal
# zijn er risico's met collect te gebruiken? wat als er heel veel data is?
# tip: gebruik enkel collect voor eindresultaten

# Voorbeeld DataFrame
df = spark.createDataFrame([('a', 1), ('b', 1), ('c', 1)], ['letter', 'count'])
df.show()

rdd collect: [('a', 1), ('b', 1), ('a', 2)]


+------+-----+
|letter|count|
+------+-----+
|     a|    1|
|     b|    1|
|     c|    1|
+------+-----+



## Data inlezen met Spark

Spark kan data inlezen uit verschillende bronnen (lokaal, cloud, ...) en formaten zoals CSV, JSON, Parquet of ORC. 
Met MinIO gebruiken we het `s3a://` protocol, waarbij Spark de Hadoop S3A connector gebruikt.

Hierbij kan je ook opties meegeven, zoals `header=True`, `inferSchema=True` enzovoort.


In [13]:
# CSV bestand uit MinIO lezen
csv_path = 's3a://02-spark/titanic.csv'
#df_csv = spark.read.option('header', True).csv(csv_path)
df_csv = spark.read.csv(csv_path, header=True, inferSchema=True)
df_csv.show(2, truncate=False)
df_csv.printSchema()

+-----------+--------+------+---------------------------------------------------+------+----+-----+-----+---------+-------+-----+--------+
|PassengerId|Survived|Pclass|Name                                               |Sex   |Age |SibSp|Parch|Ticket   |Fare   |Cabin|Embarked|
+-----------+--------+------+---------------------------------------------------+------+----+-----+-----+---------+-------+-----+--------+
|1          |0       |3     |Braund, Mr. Owen Harris                            |male  |22.0|1    |0    |A/5 21171|7.25   |NULL |S       |
|2          |1       |1     |Cumings, Mrs. John Bradley (Florence Briggs Thayer)|female|38.0|1    |0    |PC 17599 |71.2833|C85  |C       |
+-----------+--------+------+---------------------------------------------------+------+----+-----+-----+---------+-------+-----+--------+
only showing top 2 rows

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: s

## Data wegschrijven naar MinIO

Spark ondersteunt meerdere 'sinks' zoals CSV, JSON, Parquet of databases. 
Wanneer je MinIO gebruikt, specificeer je opnieuw een `s3a://` pad.

Parquet is vaak de beste keuze omdat het kolom-gebaseerd en gecomprimeerd is.
Dit is echter niet leesbaar waardoor in dit vak vaak aan een csv/json formaat de voorkeur gegeven wordt.

In [16]:
# Resultaat wegschrijven naar MinIO in Parquet formaat
output_path = 's3a://02-spark/titanic_parquet'
df_csv.write.mode('overwrite').parquet(output_path) # write is ook een actie

## Spark functies

Spark maakt gebruik van lazy evaluation voor optimalisatie van de uit te voeren code.
Om dit te doen zijn er twee soorten functies in spark

* Transformaties: Worden pas uitgevoerd wanneer het nodig is voor een actie.
* Acties: Dit is een eindpunt voor de code en er moet een eindresultaat berekend worden.

### Transformaties

Een transformatie creëert een nieuwe dataset uit een bestaande dataset, zonder dat de data direct wordt uitgevoerd. 
Voorbeelden zijn `map`, `filter`, `select`, `groupBy`.

Spark voert transformaties lui (lazy) uit, wat betekent dat de berekeningen pas gebeuren wanneer een actie wordt aangeroepen.
Merk op dat onderstaande code direct stopt zonder output.
Dat er geen berekeningen zijn merk je op doordat er geen stage-informatie uitgeprint wordt.

In [23]:
# Voorbeeld transformatie: selectie en filtering
tmp = df_csv.select('name', 'age', 'survived').filter(df_csv.Age>30)

# Acties

Acties triggeren de uitvoering van een Spark job. 
Voorbeelden zijn `collect()`, `count()`, `show()` of `write()`.

Zonder acties voert Spark niets uit, zelfs als je transformaties gedefinieerd hebt.

In [21]:
tmp.show()

+--------------------+----+--------+
|                name| age|survived|
+--------------------+----+--------+
|Cumings, Mrs. Joh...|38.0|       1|
|Futrelle, Mrs. Ja...|35.0|       1|
|Allen, Mr. Willia...|35.0|       0|
|McCarthy, Mr. Tim...|54.0|       0|
|Bonnell, Miss. El...|58.0|       1|
|Andersson, Mr. An...|39.0|       0|
|Hewlett, Mrs. (Ma...|55.0|       1|
|Vander Planke, Mr...|31.0|       0|
|Fynney, Mr. Joseph J|35.0|       0|
|Beesley, Mr. Lawr...|34.0|       1|
|Asplund, Mrs. Car...|38.0|       1|
|Uruchurtu, Don. M...|40.0|       0|
|Wheadon, Mr. Edwa...|66.0|       0|
|Holverson, Mr. Al...|42.0|       0|
|Ahlin, Mrs. Johan...|40.0|       0|
|Harper, Mrs. Henr...|49.0|       1|
|Ostby, Mr. Engelh...|65.0|       0|
| Icard, Miss. Amelie|38.0|       1|
|Harris, Mr. Henry...|45.0|       0|
|Jenkin, Mr. Steph...|32.0|       0|
+--------------------+----+--------+
only showing top 20 rows



In [24]:
# Actie: aantal records tellen
tmp.count()

305

Ga nu op zoek in de documentatie van pyspark en geef de functies die nodig zijn voor de volgende vragen op te lossen. Geef ook aan of het transformaties zijn of acties:
* Het aantal keer dat elke waarde aanwezig is in de dataset (1 functie voor wordcount uit te voeren)
* Uitfilteren van rijen
* Groeperen van een aantal rijen op basis van een bepaalde waarde.
* Toevoegen van een kolom aan elke key (bvb de lengte van een woord)
* Hoe doe je head() uit pandas op RDD's?
* Hoe doe je de apply() uit pandas op RDD's?

## Shared variables: accumulators en broadcast variabelen

Gedeelde variabelen in een gedistrubeerde/parallelle context is steeds een complex gegeven.
Hierbij komen vaak race conditions voor die opgelost worden door locks toe te voegen zodat een parameter tijdelijk geblokkeerd wordt.
Dit is echter een trage aanpak die niet geschikt is voor big-data applicaties die heel snel moeten kunnen werken.
Een oplossing is om enkel twee speciale types toe te laten: accumulators (write often, read once) en broadcast variabelen (write once, read often).

**Accumulators** zijn variabelen die door verschillende executors verhoogd kunnen worden, 
en handig zijn voor bijvoorbeeld tellingen of debugging.

Spark DataFrames hebben ook ingebouwde functies zoals `agg()` voor aggregaties, 
maar accumulators kunnen nuttig zijn bij RDD-achtige berekeningen.


In [25]:
acc = sc.accumulator(0) # sc is de sparkcontext

def count_lines(line):
    global acc
    acc += 1
    return line

rdd = sc.textFile('s3a://02-spark/input.txt')
rdd.map(count_lines).collect()
print(acc.value)

5


Een **broadcast-variabele** is een read-only kopie van data die één keer naar alle executors/workers gestuurd wordt.
In plaats van dat elke task zelf zijn eigen kopie van de data meestuurt (veel netwerk overhead!), wordt de data één keer "gebroadcast".
Zo kunnen alle taken er lokaal naar verwijzen.

Waarom is dit belangrijk?

* Efficiëntie: Stel je hebt een kleine lookup-tabel of dictionary (bv. landcodes → landnamen). Zonder broadcast zou Spark deze bij elke task meesturen → enorme overhead. Met broadcast wordt de tabel één keer verspreid → grote besparing in netwerk & geheugen.
* Prestaties bij joins: Bij een join moet de hele tabel bekeken worden. Dit is een zware operatie in gedistribueerde omgevingen. Dit kan geoptimaliseerd worden door een dataset eerst te broadcasten waardoor geen shuffle operatie meer nodig is.
```
from pyspark.sql.functions import broadcast
df_large.join(broadcast(df_small), "id").show()
```
* Controle: soms wil je als developer expliciet aangeven dat een kleine dataset moet gebroadcast worden, omdat Spark dat niet altijd automatisch goed inschat.

In [31]:
lookup = {'US': 'United States', 'BE': 'Belgium'}
broadcast_lookup = sc.broadcast(lookup)

rdd = sc.parallelize([('US', 1), ('BE', 10)])
mapped = rdd.map(lambda x: (broadcast_lookup.value[x[0]], x[1]))
mapped.collect()

[('United States', 1), ('Belgium', 10)]

## Wordcount voorbeeld

Om de api van pyspark te leren kennen kan je gaan naar de [documentatie](https://spark.apache.org/docs/latest/api/python/reference/index.html).
Een eerder stap bij stap uitleg kan je [hier](https://spark.apache.org/docs/latest/api/python/getting_started/index.html) vinden.

In onderstaande code gaan we het wordcount-voorbeeld uitwerken.

In [39]:
import pyspark.sql.functions as F
from pyspark.sql.functions import explode, split, col

input_path = 's3a://02-spark/input.txt'
text_df = spark.read.text(input_path)

text_df.show(truncate=False)

words = text_df.select(
    F.explode(
        F.split(F.col('value'), ' ')
    ).alias('word')
)
# door de tweede importlijn kan je het volgende doen, omdat je ze individueel importeert
words = text_df.select(
    explode(
        split(col('value'), ' ')
    ).alias('word')
)

words.show(5)

counts = words.groupby('word').count()

counts.show()

output_path = 's3a://02-spark/output/wordcount'
counts.write.csv(output_path, header=True, mode='overwrite')


+----------------------------------------------------------------+
|value                                                           |
+----------------------------------------------------------------+
|Hello World,                                                    |
|hello world,                                                    |
|hello world,                                                    |
|                                                                |
|Dit is een voorbeeld file om het Wordcount voorbeeld te testen !|
+----------------------------------------------------------------+

+------+
|  word|
+------+
| Hello|
|World,|
| hello|
|world,|
| hello|
+------+
only showing top 5 rows

+---------+-----+
|     word|count|
+---------+-----+
|Wordcount|    1|
|        !|    1|
|   world,|    2|
|    hello|    2|
|voorbeeld|    2|
|       is|    1|
|       te|    1|
|      Dit|    1|
|    Hello|    1|
|     file|    1|
|       om|    1|
|   testen|    1|
|      een|    1|
| 

26/03/13 16:16:54 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


**Wat gebeurt er in dit voorbeeld?**

Sparkcontext om een connectie te maken met de distributed storage
De input file wordt dan ingelezen met de read.text() functie.
Deze functie maakt een dataframe met 1 lijn per rij.
Door middel van de split functie wordt de tekst gesplitst in woorden (1 kolom met een lijst per rij/lijn).
De explode functie zorgt dan dat een lijst omgezet wordt naar meerdere rijen (1 woord per rij).
Dit kan dan gegroepeert wordt met het woord als key en de grootte van een groep is dan het aantal keer dat elk woord voorkomt.
Dit resultaat kan dan weggeschreven worden om bewaard te worden.

In de WebUI van Spark kan je de status van de huidige spark-applicatie controleren. Voor dit voorbeeld zou je iets gelijkaardig moeten kunnen zien, na het klikken op de huidige spark-session die bezig is.

![spark web ui](images/sparkwebui.png)

## Oefeningen:

Laad het input.txt bestand uit de MinIO cluster uit en voer onderstaande berekeningen uit
* Tel het aantal regels in het bestand.
* Vind het aantal woorden in het bestand.
* Maak een nieuw dataframe met alleen de regels die het woord "world" bevatten.

## PySpark SQL

Bovenstaande datastructuren (RDD's en Dataframes) zijn een onderdeel van het Pyspark sql module.
De Spark API heeft een hele reeks methoden en functies om deze in te laden, uit te lezen en te manipuleren.
Daarnaast maakt deze module het ook mogelijk om SQL-queries uit te voeren op dataframes.
Om SQL-queries uit te voeren op dataframes moet er eerst een view gemaakt worden in het dataframe met de functie createOrReplaceTempView("view_name")

Daarna kan je gebruik maken van de .sql() functie om allerhande sql queries uit te voeren.
Echter is het gebruik van deze functie iets moeilijker te debuggen omdat de hele operatie in 1 tekstuele parameter vervat is.
Om deze reden is het aangeraden om gebruik te maken van de transformaties zelf en niet van .sql() maar het werkt.

### Oefening

Een volledige lijst met alle operaties kan je [hier](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql.html#dataframe-apis) vinden.
Zoek de functies die gebruikt moeten worden om de volgende zaken uit te voeren:
* Groepeer volgens een bepaalde sleutel
* Krijg een lijst met alle kolomnamen
* Filter rijen uit
* Verwijder null-values in de dataset via rijen
* Verwijder null-values door kolommen te verwijderen
* Bereken een dataframe met statistieken van het dataframe
* Krijg een dataframe met alle nan waarden
* Hoe krijg je informatie zoals .info()
* Hoe werkt het groeperen van informatie op basis van een key/kolom

Probeer deze ook uit op bovenstaand aangemaakt dataframe

Lees daarna volgende [link](https://sparkbyexamples.com/pyspark/pyspark-aggregate-functions/) om een idee te krijgen over hoe verschillende functies uit te voeren op deze dataframes.
Werk nu de volgende oefening uit en maak hiervoor een spark applicatie:
* Download de iris dataset in sklearn
* Bewaar deze dataset als csv en upload de file naar MinIO
* Schrijf de code om de csv uit te lezen en om te zetten naar een dataframe
* Print het dataschema uit voor het dataframe, hoeveel kolommen zijn er aanwezig in het dataframe.
* Bereken het minimum en maximum van de 'sepal width (cm)' en 'petal width (cm)' kolom.
* Hernoem de target kolom naar label
* Hernoem de labels 0 naar Soort 0, labels 1 naar Soort 2 en labels 3 naar Soort 3
* Voer normalisatie van de eerste 4 kolommen uit (het gemiddelde ervan aftrekken en delen door de standaardafwijking)
* Controleer de voorgaande stap door opnieuw het gemiddelde en de standaardafwijking te berekenen. Deze moeten respectievelijk 0 en 1 zijn.
* Bereken de oppervlakte van sepal door de lengte en breedte ervan te vermenigvuldigen. Noem deze nieuwe kolom sepal area. Doe dit ook voor de petal.
* Groepeer nu de rijen op de label kolom. Bereken per groep het gemiddelde van elke kolom. Is er een verschil tussen de verschillende klassen?

## Oefeningen

Maak een pyspark DataFrame van onderstaande lijst met dictionaries en voer volgende operaties uit:
* Toon de schema van het DataFrame.
* Filter het DataFrame om alleen de rijen te behouden waar de leeftijd groter is dan 28.

Laad het input.csv CSV-bestand in een pyspark DataFrame en voer volgende operaties uit:
* Toon de eerste 3 rijen van het DataFrame.
* Bereken de gemiddelde sepal length in het DataFrame.
* Groepeer het DataFrame op basis van het target en tel het aantal per klasse.

Voer met onderstaande data de volgende stappen uit:
* Maak een pyspark DataFrame van de onderstaande lijst.
* Voeg een nieuwe kolom age_after_5_years toe die de leeftijd over 5 jaar toont.
* Filter het DataFrame om alleen de rijen te behouden waar de leeftijd na 5 jaar groter is dan 30.

Gebruik acties om de volgende resultaten uit het DataFrame van de vorige stap te halen door de volgende stapen te implementeren
* Gebruik de actie count() om het aantal rijen in het DataFrame te tellen.
* Gebruik de actie show() om de inhoud van het DataFrame te tonen.
* Selecteer alleen de name en age kolommen en toon de resultaten.

Voer de volgende aggregaties uit op een pyspark DataFrame op basis van onderstaande data:
* Groepeer het DataFrame op basis van de afdeling en bereken het gemiddelde salaris per afdeling.
* Sorteer de resultaten op basis van het gemiddelde salaris in aflopende volgorde.

## Verdere oefeningen

Gebruik nu de titanic csv om met behulp van een spark applicatie de volgende zaken te berekenen:
* Het aantal unieke plaatsen waar personen aan boord zijn gekomen (embarked kolom)
* Het aantal ontbrekende waarden in de Cabin kolom
* De volgende statistische waarden voor de ticketprijs (Fare) kolom: min, max, mean, std
* De langste naam van een passagier
* Het aantal passagiers per klasse
* Het totaal aantal passagiers op de titanic

Het is de bedoeling dat je elk element afzonderlijk berekend dus je moet geen dataframe uitkomen waar al deze zaken in zitten. 

Tip: herstart je kernel zodat je het opzetten van de sparkcontext ook oefent. 